# [Optional] Introduction to Strands Agents

This optional module introduces the Strands Agents SDK for building AI agents. Complete this if you're new to agentic AI development before proceeding to the main Developer Journey labs.

## Learning Objectives

By the end of this notebook, you will be able to:
- Create a basic agent using the Strands Agents SDK
- Add built-in and custom tools to extend agent capabilities
- Understand the agent execution loop and metrics
- Deploy an agent to Amazon Bedrock AgentCore Runtime

**Duration**: ~30 minutes

## What is Strands Agents?

[Strands Agents](https://strandsagents.com/) is an open-source SDK for building AI agents with a model-driven approach. Instead of manually orchestrating complex workflows, Strands lets the model decide when and how to use tools.

### Core Components

| Component | Description |
|-----------|-------------|
| **Model** | The LLM that powers reasoning (e.g., Claude on Bedrock) |
| **System Prompt** | Instructions defining agent behavior and personality |
| **Tools** | Functions the agent can call to take actions |

### Agent Loop Architecture

![Strands Agent Architecture](images/strands-agent-architecture.png)

The agent autonomously decides whether to respond directly or use tools based on the user's request.

---

## 1. Setup

Configure AWS credentials and model settings.

In [1]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Configuration
REGION = os.getenv('AWS_DEFAULT_REGION', 'us-east-1')
MODEL_ID = "us.anthropic.claude-sonnet-4-6"

print(f"Region: {REGION}")
print(f"Model: {MODEL_ID}")

Region: us-east-1
Model: us.anthropic.claude-sonnet-4-6


---

## 2. Your First Agent

Create a simple agent with just a few lines of code. The agent uses Claude on Amazon Bedrock for reasoning.

In [2]:
from strands import Agent

# Create a basic agent
agent = Agent(
    model=MODEL_ID,
    system_prompt="You are a helpful assistant. Be concise and accurate."
)

# Invoke the agent
agent("What is Amazon Bedrock in one sentence?")

Amazon Bedrock is a fully managed AWS service that provides access to foundation models (FMs) from various AI companies through a single API, enabling developers to build and scale generative AI applications without managing infrastructure.

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': 'Amazon Bedrock is a fully managed AWS service that provides access to foundation models (FMs) from various AI companies through a single API, enabling developers to build and scale generative AI applications without managing infrastructure.'}]}, metrics=EventLoopMetrics(cycle_count=1, tool_metrics={}, cycle_durations=[3.1890218257904053], agent_invocations=[AgentInvocation(cycles=[EventLoopCycleMetric(event_loop_cycle_id='bc651bb2-91fc-48fc-aa63-bb1e3bafc4fb', usage={'inputTokens': 30, 'outputTokens': 46, 'totalTokens': 76})], usage={'inputTokens': 30, 'outputTokens': 46, 'totalTokens': 76})], traces=[<strands.telemetry.metrics.Trace object at 0x10f342650>], accumulated_usage={'inputTokens': 30, 'outputTokens': 46, 'totalTokens': 76}, accumulated_metrics={'latencyMs': 1896}), state={}, interrupts=None, structured_output=None)

<div class="alert alert-block alert-info">
<b>Note:</b> The agent automatically handles the conversation with the model. You simply call the agent like a function with your input.
</div>

---

## 3. Adding Tools

Tools extend agent capabilities beyond text generation. The agent autonomously decides when to use tools based on the user's request.

### Built-in Tools

Strands provides common tools out of the box via the `strands-agents-tools` package:

In [3]:
from strands import Agent
from strands_tools import calculator, current_time

# Create agent with built-in tools
agent_with_tools = Agent(
    model=MODEL_ID,
    tools=[calculator, current_time],
    system_prompt="You are a helpful assistant with access to a calculator and clock."
)

# The agent automatically uses the calculator tool
agent_with_tools("What is 15% of 250?")


Tool #1: calculator


╭────────────────────────────────────────────── Calculation Result ───────────────────────────────────────────────╮
│                                                                                                                 │
│  ╭───────────┬─────────────────────╮                                                                            │
│  │ Operation │ Evaluate Expression │                                                                            │
│  │ Input     │ 0.15 * 250          │                                                                            │
│  │ Result    │ 37.5                │                                                                            │
│  ╰───────────┴─────────────────────╯                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

**15% of 250 is 37.5**

Here's the simple breakdown:
- 15% = 0.15
- 0.15 × 250 = **37.5**

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': "**15% of 250 is 37.5**\n\nHere's the simple breakdown:\n- 15% = 0.15\n- 0.15 × 250 = **37.5**"}]}, metrics=EventLoopMetrics(cycle_count=2, tool_metrics={'calculator': ToolMetrics(tool={'toolUseId': 'tooluse_zbnFHc3TB7hxsBEnZs70Zt', 'name': 'calculator', 'input': {'expression': '0.15 * 250'}}, call_count=1, success_count=1, error_count=0, total_time=0.007370948791503906)}, cycle_durations=[2.0020718574523926], agent_invocations=[AgentInvocation(cycles=[EventLoopCycleMetric(event_loop_cycle_id='8ce1a5b2-3c3c-45b5-b3e1-34a6ed37e8fa', usage={'inputTokens': 2053, 'outputTokens': 58, 'totalTokens': 2111}), EventLoopCycleMetric(event_loop_cycle_id='176c1f5d-2897-4510-a9f9-b4870b9426b1', usage={'inputTokens': 2129, 'outputTokens': 50, 'totalTokens': 2179})], usage={'inputTokens': 4182, 'outputTokens': 108, 'totalTokens': 4290})], traces=[<strands.telemetry.metrics.Trace object at 0x10fe781d0>, <strands.tele

In [4]:
# The agent uses the current_time tool
agent_with_tools("What time is it now?")


Tool #2: current_time
The current time is **4:09 AM UTC** on **June 1, 2026**. 🕓

If you'd like the time in a specific timezone, just let me know!

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': "The current time is **4:09 AM UTC** on **June 1, 2026**. 🕓\n\nIf you'd like the time in a specific timezone, just let me know!"}]}, metrics=EventLoopMetrics(cycle_count=4, tool_metrics={'calculator': ToolMetrics(tool={'toolUseId': 'tooluse_zbnFHc3TB7hxsBEnZs70Zt', 'name': 'calculator', 'input': {'expression': '0.15 * 250'}}, call_count=1, success_count=1, error_count=0, total_time=0.007370948791503906), 'current_time': ToolMetrics(tool={'toolUseId': 'tooluse_CdzOuJaV1eYqcpThuCYnDA', 'name': 'current_time', 'input': {}}, call_count=1, success_count=1, error_count=0, total_time=0.0016222000122070312)}, cycle_durations=[2.0020718574523926, 2.2362101078033447], agent_invocations=[AgentInvocation(cycles=[EventLoopCycleMetric(event_loop_cycle_id='8ce1a5b2-3c3c-45b5-b3e1-34a6ed37e8fa', usage={'inputTokens': 2053, 'outputTokens': 58, 'totalTokens': 2111}), EventLoopCycleMetric(event_loop_cycle_id='176c1f5d-

### Custom Tools

Create domain-specific tools using the `@tool` decorator. This is essential for building agents that interact with your business systems.

In [5]:
from strands import Agent, tool

@tool
def lookup_order(order_id: str) -> str:
    """Look up order status by order ID.
    
    Args:
        order_id: The order ID to look up (e.g., ORD-12345)
    
    Returns:
        Order status information
    """
    # Simulated order database
    orders = {
        "ORD-12345": {"status": "Shipped", "eta": "2 days", "carrier": "FedEx"},
        "ORD-67890": {"status": "Processing", "eta": "5 days", "carrier": "Pending"},
    }
    
    if order_id in orders:
        order = orders[order_id]
        return f"Order {order_id}: {order['status']}, ETA: {order['eta']}, Carrier: {order['carrier']}"
    return f"Order {order_id} not found"


@tool
def check_inventory(product_sku: str) -> str:
    """Check inventory for a product SKU.
    
    Args:
        product_sku: The product SKU to check
    
    Returns:
        Inventory status
    """
    # Simulated inventory
    inventory = {
        "SKU-KB-001": {"name": "Wireless Keyboard", "qty": 150, "warehouse": "Seattle"},
        "SKU-MS-002": {"name": "Gaming Mouse", "qty": 0, "warehouse": "N/A"},
    }
    
    if product_sku in inventory:
        item = inventory[product_sku]
        status = "In Stock" if item['qty'] > 0 else "Out of Stock"
        return f"{item['name']}: {status} ({item['qty']} units in {item['warehouse']})"
    return f"Product {product_sku} not found"


print("Custom tools defined: lookup_order, check_inventory")

Custom tools defined: lookup_order, check_inventory


<div class="alert alert-block alert-warning">
<b>Key Insight:</b> The docstring in your tool function is critical - it tells the model what the tool does and when to use it. Write clear, descriptive docstrings for better tool selection.
</div>

---

## 4. Customer Support Agent Example

Let's build a customer support agent similar to what we'll use in the main Developer Journey labs.

In [6]:
support_agent = Agent(
    model=MODEL_ID,
    tools=[lookup_order, check_inventory],
    system_prompt="""You are a customer support agent for CloudCommerce.

Your responsibilities:
- Help customers check order status using the lookup_order tool
- Check product inventory using the check_inventory tool
- Be friendly, professional, and helpful

Always use the available tools to get accurate information.
If you cannot find information, apologize and offer alternatives."""
)

print("Customer support agent created with 2 tools")

Customer support agent created with 2 tools


In [7]:
# Test order lookup
support_agent("Where is my order ORD-12345?")

Let me look that up for you right away!
Tool #1: lookup_order
Great news! Here's the status of your order **ORD-12345**:

- 📦 **Status:** Shipped
- 🚚 **Carrier:** FedEx
- 📅 **Estimated Arrival:** 2 days

Your order is on its way! If you need the tracking number or have any other questions, feel free to ask. 😊

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': "Great news! Here's the status of your order **ORD-12345**:\n\n- 📦 **Status:** Shipped\n- 🚚 **Carrier:** FedEx\n- 📅 **Estimated Arrival:** 2 days\n\nYour order is on its way! If you need the tracking number or have any other questions, feel free to ask. 😊"}]}, metrics=EventLoopMetrics(cycle_count=2, tool_metrics={'lookup_order': ToolMetrics(tool={'toolUseId': 'tooluse_AKDSjGXSg4kuaMkfPBDn38', 'name': 'lookup_order', 'input': {'order_id': 'ORD-12345'}}, call_count=1, success_count=1, error_count=0, total_time=0.0015230178833007812)}, cycle_durations=[3.0851900577545166], agent_invocations=[AgentInvocation(cycles=[EventLoopCycleMetric(event_loop_cycle_id='a0f33f85-68c8-4f48-abd2-edf6b9980365', usage={'inputTokens': 767, 'outputTokens': 71, 'totalTokens': 838}), EventLoopCycleMetric(event_loop_cycle_id='87ec9131-8870-444a-a93e-b5c71b8b3829', usage={'inputTokens': 875, 'outputTokens': 91, 'totalTokens': 

In [8]:
# Test inventory check
support_agent("Do you have the wireless keyboard SKU-KB-001 in stock?")

Let me check that for you right now!
Tool #2: check_inventory
Great news! Here's the inventory status for the **Wireless Keyboard (SKU-KB-001)**:

- ✅ **Status:** In Stock
- 📦 **Available Units:** 150
- 📍 **Location:** Seattle

The wireless keyboard is readily available, so you should have no trouble ordering one! Would you like help with anything else? 😊

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': "Great news! Here's the inventory status for the **Wireless Keyboard (SKU-KB-001)**:\n\n- ✅ **Status:** In Stock\n- 📦 **Available Units:** 150\n- 📍 **Location:** Seattle\n\nThe wireless keyboard is readily available, so you should have no trouble ordering one! Would you like help with anything else? 😊"}]}, metrics=EventLoopMetrics(cycle_count=4, tool_metrics={'lookup_order': ToolMetrics(tool={'toolUseId': 'tooluse_AKDSjGXSg4kuaMkfPBDn38', 'name': 'lookup_order', 'input': {'order_id': 'ORD-12345'}}, call_count=1, success_count=1, error_count=0, total_time=0.0015230178833007812), 'check_inventory': ToolMetrics(tool={'toolUseId': 'tooluse_o1I4a6heKeQycWJp81nc8A', 'name': 'check_inventory', 'input': {'product_sku': 'SKU-KB-001'}}, call_count=1, success_count=1, error_count=0, total_time=0.0008227825164794922)}, cycle_durations=[3.0851900577545166, 2.6261887550354004], agent_invocations=[AgentInvocation(c

In [9]:
# Test handling unknown order
support_agent("What's the status of order ORD-99999?")

Let me check that order for you right away!
Tool #3: lookup_order
I'm sorry, but I wasn't able to find any order with the ID **ORD-99999**. 😕

Here are a few suggestions:
- **Double-check the order ID** – Make sure the number is correct (it may be in your confirmation email).
- **Check your account** – Log into your CloudCommerce account to view your order history.
- **Contact us directly** – If you're still having trouble, we can escalate this to our support team for further investigation.

Is there anything else I can help you with? 😊

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': "I'm sorry, but I wasn't able to find any order with the ID **ORD-99999**. 😕\n\nHere are a few suggestions:\n- **Double-check the order ID** – Make sure the number is correct (it may be in your confirmation email).\n- **Check your account** – Log into your CloudCommerce account to view your order history.\n- **Contact us directly** – If you're still having trouble, we can escalate this to our support team for further investigation.\n\nIs there anything else I can help you with? 😊"}]}, metrics=EventLoopMetrics(cycle_count=6, tool_metrics={'lookup_order': ToolMetrics(tool={'toolUseId': 'tooluse_GfUlr2rzcW8h7Gd4tLiuZ5', 'name': 'lookup_order', 'input': {'order_id': 'ORD-99999'}}, call_count=2, success_count=2, error_count=0, total_time=0.002379894256591797), 'check_inventory': ToolMetrics(tool={'toolUseId': 'tooluse_o1I4a6heKeQycWJp81nc8A', 'name': 'check_inventory', 'input': {'product_sku': 'SKU-KB-001

---

## 5. Understanding Agent Execution

After running an agent, you can inspect execution metrics to understand what happened.

In [10]:
result = support_agent("Check if SKU-MS-002 is available")

# Access metrics from the result
print("\n" + "=" * 50)
print("EXECUTION METRICS")
print("=" * 50)

if hasattr(result, 'metrics'):
    metrics = result.metrics
    if hasattr(metrics, 'accumulated_usage'):
        usage = metrics.accumulated_usage
        print(f"Input tokens:  {usage.get('inputTokens', 'N/A')}")
        print(f"Output tokens: {usage.get('outputTokens', 'N/A')}")
        print(f"Total tokens:  {usage.get('totalTokens', 'N/A')}")
else:
    print("Metrics not available in this response format")

Sure, let me check that for you!
Tool #4: check_inventory
Here's the inventory status for **SKU-MS-002**:

- ❌ **Status:** Out of Stock
- 📦 **Available Units:** 0

Unfortunately, the **Gaming Mouse** is currently out of stock. Here are some suggestions:
- **Check back later** – Stock is often replenished regularly.
- **Browse alternatives** – We may have similar products available. Let me know if you'd like me to check another SKU!
- **Contact our team** – We can notify you when this item becomes available again.

Is there anything else I can help you with? 😊
EXECUTION METRICS
Input tokens:  9106
Output tokens: 728
Total tokens:  9834


---

## 6. Deploy to Amazon Bedrock AgentCore Runtime

Amazon Bedrock AgentCore Runtime provides a managed, serverless environment for running agents in production.

![AgentCore Runtime Architecture](images/agentcore-runtime-architecture.png)

### What is AgentCore Runtime?

AgentCore Runtime is a fully managed service that hosts your AI agents:

1. **Packages your code** into a container
2. **Creates AWS resources** (IAM roles, ECR, etc.)
3. **Hosts your agent** in a serverless environment
4. **Provides an API endpoint** for invocation

### Key Benefits

| Feature | Description |
|---------|-------------|
| **Serverless** | No infrastructure to manage |
| **Auto-scaling** | Handles variable load automatically |
| **Session Isolation** | Each conversation runs in isolated context |
| **Framework Agnostic** | Works with Strands, LangGraph, CrewAI, and more |

### Step 1: Create Agent File

Create a file called `demo_agent.py` with your agent wrapped in `BedrockAgentCoreApp`:

In [11]:
%%writefile demo_agent.py
from bedrock_agentcore import BedrockAgentCoreApp
from strands import Agent, tool

app = BedrockAgentCoreApp()

@tool
def lookup_order(order_id: str) -> str:
    """Look up order status by order ID."""
    orders = {
        "ORD-12345": {"status": "Shipped", "eta": "2 days", "carrier": "FedEx"},
        "ORD-67890": {"status": "Processing", "eta": "5 days", "carrier": "Pending"},
    }
    if order_id in orders:
        order = orders[order_id]
        return f"Order {order_id}: {order['status']}, ETA: {order['eta']}, Carrier: {order['carrier']}"
    return f"Order {order_id} not found"

agent = Agent(
    model="us.anthropic.claude-sonnet-4-6",
    tools=[lookup_order],
    system_prompt="You are a helpful customer support agent."
)

@app.entrypoint
def invoke(payload):
    response = agent(payload.get("prompt", ""))
    return {"result": response.message}

if __name__ == "__main__":
    app.run()

Writing demo_agent.py


In [12]:
%%writefile requirements-demo.txt
bedrock-agentcore
strands-agents

Writing requirements-demo.txt


### Step 2: Configure and Deploy

Use the AgentCore Starter Toolkit to configure and deploy the agent:

In [13]:
# Guard: remove stale AgentCore config so this lab can't reuse another lab's agent.
# This directory shares one .bedrock_agentcore.yaml across notebooks (e.g. 04-llm-routing
# deploys 'customer_support_v4_routing'). A leftover config makes launch() reuse the wrong
# runtime/ECR image, so the deployed agent serves different code than demo_agent.py.
import os

stale = ".bedrock_agentcore.yaml"
if os.path.exists(stale):
    os.remove(stale)
    print(f"Removed stale config: {stale}")
else:
    print("No stale config found - clean start")

No stale config found - clean start


In [14]:
from bedrock_agentcore_starter_toolkit import Runtime

# Initialize the Runtime
agentcore_runtime = Runtime()
AGENT_NAME = "demo_agent"

# Configure the agent
agentcore_runtime.configure(
    entrypoint="demo_agent.py",
    requirements_file="requirements-demo.txt",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    region=REGION,
    agent_name=AGENT_NAME
)

print("Agent configured")

/Users/tracilim/Projects/aws-bedrock-prompt-optimization-workshop/.venv/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.1)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(
Entrypoint parsed: file=/Users/tracilim/Projects/aws-bedrock-prompt-optimization-workshop/02-developer-journey/demo_agent.py, bedrock_agentcore_name=demo_agent
Memory disabled - agent will be stateless
Configuring BedrockAgentCore agent: demo_agent


💡 No container engine found (Docker/Finch/Podman not installed)

✓ Default deployment uses CodeBuild (no container engine needed), For local builds, install Docker, Finch, or 
Podman

Memory disabled
Network mode: PUBLIC


📄 Generated Dockerfile: 
/Users/tracilim/Projects/aws-bedrock-prompt-optimization-workshop/02-developer-journey/Dockerfile

Generated .dockerignore: /Users/tracilim/Projects/aws-bedrock-prompt-optimization-workshop/02-developer-journey/.dockerignore
Setting 'demo_agent' as default agent
Bedrock AgentCore configured: /Users/tracilim/Projects/aws-bedrock-prompt-optimization-workshop/02-developer-journey/.bedrock_agentcore.yaml


Agent configured


### Step 3: Deploy and Test

Deploy the agent and test it:

In [15]:
# Deploy to AgentCore Runtime
# This builds the container, pushes to ECR, and creates the runtime
launch_result = agentcore_runtime.launch(auto_update_on_conflict=True)
print(f"Agent deployed: {launch_result.agent_arn}")

🚀 Launching Bedrock AgentCore (cloud mode - RECOMMENDED)...
   • Deploy Python code directly to runtime
   • No Docker required (DEFAULT behavior)
   • Production-ready deployment

💡 Deployment options:
   • runtime.launch()                → Cloud (current)
   • runtime.launch(local=True)      → Local development
Memory disabled - skipping memory creation
Starting CodeBuild ARM64 deployment for agent 'demo_agent' to account 739907928487 (us-east-1)
Generated image tag: 20260601-041005-106
Setting up AWS resources (ECR repository, execution roles)...
Getting or creating ECR repository for agent: demo_agent


Repository doesn't exist, creating new ECR repository: bedrock-agentcore-demo_agent


ECR repository available: 739907928487.dkr.ecr.us-east-1.amazonaws.com/bedrock-agentcore-demo_agent
Getting or creating execution role for agent: demo_agent
Using AWS region: us-east-1, account ID: 739907928487
Role name: AmazonBedrockAgentCoreSDKRuntime-us-east-1-20b5f127ba
Role doesn't exist, creating new execution role: AmazonBedrockAgentCoreSDKRuntime-us-east-1-20b5f127ba
Starting execution role creation process for agent: demo_agent
✓ Role creating: AmazonBedrockAgentCoreSDKRuntime-us-east-1-20b5f127ba
Creating IAM role: AmazonBedrockAgentCoreSDKRuntime-us-east-1-20b5f127ba
✓ Role created: arn:aws:iam::739907928487:role/AmazonBedrockAgentCoreSDKRuntime-us-east-1-20b5f127ba
✓ Execution policy attached: BedrockAgentCoreRuntimeExecutionPolicy-demo_agent
Role creation complete and ready for use with Bedrock AgentCore
Execution role available: arn:aws:iam::739907928487:role/AmazonBedrockAgentCoreSDKRuntime-us-east-1-20b5f127ba
Preparing CodeBuild project and uploading source...
Getting

Agent deployed: arn:aws:bedrock-agentcore:us-east-1:739907928487:runtime/demo_agent-usLRZW7KhM


In [16]:
import time

# Wait for agent to be ready
status = agentcore_runtime.status()
agent_status = status.endpoint["status"]
terminal_states = {"READY", "CREATE_FAILED", "DELETE_FAILED", "UPDATE_FAILED"}

while agent_status not in terminal_states:
    print(f"Agent status: {agent_status}")
    time.sleep(10)
    status = agentcore_runtime.status()
    agent_status = status.endpoint["status"]

print(f"Final status: {agent_status}")

Retrieved Bedrock AgentCore status for: demo_agent


Final status: READY


In [17]:
import os
import json
import uuid
import boto3

region = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")
data_client = boto3.client("bedrock-agentcore", region_name=region)

# Get agent ARN from deployment result or runtime status
try:
    agent_arn = launch_result.agent_arn
except NameError:
    status = agentcore_runtime.status()
    agent_arn = status.runtime["agentRuntimeArn"]


def invoke_agent(prompt):
    """Invoke the agent via AgentCore API."""
    response = data_client.invoke_agent_runtime(
        agentRuntimeArn=agent_arn,
        runtimeSessionId=str(uuid.uuid4()),
        payload=json.dumps({"prompt": prompt}).encode(),
    )
    return json.loads(response["response"].read().decode("utf-8"))


# Test the deployed agent
print(f"Agent ARN: {agent_arn}")
result = invoke_agent("where is my order ORD-12345?")
print(result)

# Assert the runtime is serving demo_agent.py, not another lab's agent.
# demo_agent.py returns {"result": ...}; a stale routing agent returns
# {"response", "model_used", "complexity"}. Fail loudly on mismatch.
assert "result" in result, (
    f"Deployed agent returned unexpected shape {list(result.keys())} - "
    "the runtime is serving a different agent than demo_agent.py. "
    "Re-run the guard cell to clear stale config, then redeploy."
)
assert "ORD-12345" in str(result["result"]), (
    "Agent did not return order ORD-12345 - the lookup_order tool is missing. "
    "The wrong container is deployed; clear stale config and redeploy."
)
print("\n✅ Verified: runtime is serving demo_agent.py with the lookup_order tool")

Agent ARN: arn:aws:bedrock-agentcore:us-east-1:739907928487:runtime/demo_agent-usLRZW7KhM
{'result': {'role': 'assistant', 'content': [{'text': "Great news! Here's the status of your order **ORD-12345**:\n\n- **Status:** Shipped ✅\n- **Carrier:** FedEx\n- **Estimated Delivery:** 2 days\n\nYour order is on its way! You may want to check the FedEx website for more detailed tracking information. Is there anything else I can help you with?"}], 'metadata': {'usage': {'inputTokens': 699, 'outputTokens': 87, 'totalTokens': 786}, 'metrics': {'latencyMs': 4396, 'timeToFirstByteMs': 1976}}}}

✅ Verified: runtime is serving demo_agent.py with the lookup_order tool


In [18]:
# Clean up AgentCore resources using Python API
agentcore_runtime.destroy(delete_ecr_repo=True)

# Clean up local files
import os
for f in ['demo_agent.py', 'requirements-demo.txt', '.bedrock_agentcore.yaml', 'Dockerfile', '.dockerignore']:
    if os.path.exists(f):
        os.remove(f)

print("Cleanup complete!")

🗑️ Destroying Bedrock AgentCore resources
   • Including ECR repository deletion
Starting destroy operation for agent: demo_agent (dry_run=False, delete_ecr_repo=True)
DEFAULT endpoint will be automatically deleted with agent
Deleted AgentCore agent: arn:aws:bedrock-agentcore:us-east-1:739907928487:runtime/demo_agent-usLRZW7KhM
Checking ECR repository: bedrock-agentcore-demo_agent in region: us-east-1
Deleted 1 ECR images from bedrock-agentcore-demo_agent
Deleted ECR repository: bedrock-agentcore-demo_agent
Deleted CodeBuild project: bedrock-agentcore-demo_agent-builder
Deleted S3 artifact: demo_agent/deployment.zip
Deleted S3 artifact: demo_agent/source.zip
Deleted inline policy CodeBuildExecutionPolicy from role AmazonBedrockAgentCoreSDKCodeBuild-us-east-1-20b5f127ba
Deleted CodeBuild IAM role: AmazonBedrockAgentCoreSDKCodeBuild-us-east-1-20b5f127ba
Deleted IAM role: AmazonBedrockAgentCoreSDKRuntime-us-east-1-20b5f127ba
Removed agent configuration: demo_agent
Cleared default agent (n

Cleanup complete!


---

## Summary

| Concept | Description |
|---------|-------------|
| **Agent** | Model + System Prompt + Tools |
| **Tools** | Functions the agent can call |
| **@tool decorator** | Creates custom tools from Python functions |
| **Agent Loop** | Reason -> Select Tool -> Execute -> Repeat/Respond |
| **AgentCore Runtime** | Managed serverless environment for production |

### Key Takeaways

1. **Simple API**: Create agents with just 3 components (model, prompt, tools)
2. **Tool-driven**: The model decides when and how to use tools
3. **Custom tools**: Use `@tool` decorator for domain-specific functionality
4. **Production-ready**: AgentCore Runtime handles scaling and infrastructure

---

> **Note:** The tools used in this introduction (`lookup_order`, `check_inventory`) are different from the tools used in the main Developer Journey labs (`get_return_policy`, `get_product_info`, `get_technical_support`). The concepts you learned here — creating agents, defining custom tools, and deploying to AgentCore — apply the same way throughout the workshop.

**Next**: Continue to the main Developer Journey labs to build a complete customer support system with observability and evaluation.